# Kaggle V3C: tải embedding + map, build FAISS, chạy API

Bật **Internet** và **GPU**. Gắn `rclone.conf` tại đường dẫn bên dưới hoặc tạo
Kaggle Secret `RCLONE_CONFIG` chứa nội dung file. Tạo Secret `NGROK_AUTHTOKEN`.
Chạy các cell theo thứ tự. Notebook clone nhánh `vbs-dev`, chỉ gọi pipeline có
sẵn của repo; không sửa mã nguồn trong bản clone.

Dữ liệu sau giải nén nằm trong `/kaggle/working/event-retrieval-system/data`, với
cùng cây thư mục như local. Cần đẩy bản config V3C hiện tại lên `vbs-dev` trước
khi chạy notebook. Kaggle chỉ giữ server hoạt động khi phiên notebook còn chạy.


In [ ]:
# 1. Cấu hình và clone repo
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/iamtro-2006/event-retrieval-system.git"
REPO_BRANCH = "vbs-dev"
REPO_DIR = Path("/kaggle/working/event-retrieval-system")
BACKEND_DIR = REPO_DIR / "backend"
EMBEDDING_FOLDER_ID = "1gLyt3E6w6AdNh6YKxO5Zqn_Nj6zlwFfq"
MAP_FOLDER_ID = "1N4dWx4Od2bgazMf6_PEj0TKDR5t2LneD"
RCLONE_CONFIG_SOURCE = Path("/kaggle/input/datasets/vnnoname/rclone-new/rclone.conf")
WORK_DIR = Path("/kaggle/tmp/v3c_server")
WORK_DIR.mkdir(parents=True, exist_ok=True)

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch", REPO_BRANCH,
                    REPO_URL, str(REPO_DIR)], check=True)
assert (BACKEND_DIR / "scripts/retrieval/run.py").is_file(), "Repo clone chưa hoàn chỉnh"

import yaml
app_path = BACKEND_DIR / "configs/app.yaml"
app_text = app_path.read_text(encoding="utf-8")
old_index = "index_dir: ../data/database/faiss_qwen3_vl_embedding_2b\n"
new_index = "index_dir: ../data/database/faiss_qwen3_vl_embedding_2b_vllm\n"
if old_index in app_text:
    app_text = app_text.replace(old_index, new_index, 1)
    app_path.write_text(app_text, encoding="utf-8")
    print("Đã đồng bộ index_dir trong app.yaml của bản clone")
app_cfg = yaml.safe_load(app_text)
index_cfg = yaml.safe_load((BACKEND_DIR / "configs/indexing.yaml").read_text(encoding="utf-8"))
dataset_cfg = app_cfg["datasets"]["available"].get("v3c")
expected_index = "../data/database/faiss_qwen3_vl_embedding_2b_vllm"
assert dataset_cfg is not None, "Nhánh vbs-dev chưa có config V3C; hãy đẩy thay đổi local lên remote."
assert dataset_cfg["semantic_models"]["qwen3-vl-embedding-2b"]["index_dir"] == expected_index
assert index_cfg["datasets"]["available"]["v3c"]["index"]["output_dir"] == expected_index
assert app_cfg["semantic"]["models"][0]["model_extra"]["engine"] == "vllm"
print("Repo/config V3C đã sẵn sàng:", REPO_DIR)


In [ ]:
# 2. Cài dependency cho index/API và rclone; chuẩn bị config Drive
import sys
import urllib.request
import zipfile

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu", "fastapi",
                "faster-whisper", "httpx", "numba", "pandas", "python-multipart", "PyYAML",
                "requests", "Pillow", "elasticsearch", "tqdm", "uvicorn", "pyngrok", "uv"], check=True)

rclone = shutil.which("rclone")
if not rclone:
    release = "v1.75.2"
    archive = WORK_DIR / f"rclone-{release}-linux-amd64.zip"
    url = f"https://github.com/rclone/rclone/releases/download/{release}/{archive.name}"
    urllib.request.urlretrieve(url, archive)
    with zipfile.ZipFile(archive) as zf:
        member = next(m for m in zf.namelist() if m.endswith("/rclone"))
        rclone = WORK_DIR / "rclone"
        rclone.write_bytes(zf.read(member))
        rclone.chmod(0o755)
    rclone = str(rclone)

rclone_config = WORK_DIR / "rclone.conf"
config_secret = secret("RCLONE_CONFIG")
if config_secret:
    rclone_config.write_text(config_secret, encoding="utf-8")
elif RCLONE_CONFIG_SOURCE.is_file():
    shutil.copy2(RCLONE_CONFIG_SOURCE, rclone_config)
else:
    raise FileNotFoundError("Gắn rclone.conf hoặc tạo Kaggle Secret RCLONE_CONFIG")
rclone_config.chmod(0o600)
subprocess.run([rclone, "--config", str(rclone_config), "listremotes"], check=True)


In [ ]:
# 3. Tải đúng 00001–00100.zip và 00001–00100.csv; giải nén vào cây thư mục local
EMBED_DIR = REPO_DIR / "data/embeddings/embeddings_qwen3_vl_embedding_2b_vllm/V3C"
MAP_DIR = REPO_DIR / "data/map_keyframes/V3C"
ZIP_DIR = WORK_DIR / "embedding_zips"
EMBED_DIR.mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)
ZIP_DIR.mkdir(parents=True, exist_ok=True)
(REPO_DIR / "data/keyframes/V3C").mkdir(parents=True, exist_ok=True)
(REPO_DIR / "data/videos/V3C").mkdir(parents=True, exist_ok=True)

def drive_files(folder_id):
    result = subprocess.run([rclone, "--config", str(rclone_config),
                             "--drive-root-folder-id", folder_id, "lsf", "iamtro:",
                             "--files-only", "--format", "p"], check=True,
                            capture_output=True, text=True)
    return set(result.stdout.splitlines())

ids = [f"{i:05d}" for i in range(1, 101)]
embedding_files = drive_files(EMBEDDING_FOLDER_ID)
map_files = drive_files(MAP_FOLDER_ID)
missing_zips = [name + ".zip" for name in ids if name + ".zip" not in embedding_files]
if missing_zips:
    raise FileNotFoundError("Thiếu ZIP embedding: " + ", ".join(missing_zips))
missing_maps = [name + ".csv" for name in ids if name + ".csv" not in map_files]
if missing_maps:
    print("Thiếu map CSV (index vẫn build, timestamp các video này có thể thiếu):", missing_maps)

for video_id in ids:
    target = EMBED_DIR / video_id
    if not list(target.glob("*.npy")):
        zip_path = ZIP_DIR / f"{video_id}.zip"
        subprocess.run([rclone, "--config", str(rclone_config),
                        "--drive-root-folder-id", EMBEDDING_FOLDER_ID, "copyto",
                        f"iamtro:{video_id}.zip", str(zip_path), "-P"], check=True)
        with zipfile.ZipFile(zip_path) as zf:
            members = [m for m in zf.infolist() if not m.is_dir() and m.filename.endswith(".npy")]
            if not members:
                raise ValueError(f"ZIP rỗng: {zip_path}")
            target.mkdir(parents=True, exist_ok=True)
            for member in members:
                # ZIP có thể chứa <video_id>/<frame>.npy hoặc chỉ <frame>.npy.
                # Chỉ nhận tên frame, tránh path traversal và thư mục lồng đôi.
                frame_name = Path(member.filename).name
                with zf.open(member) as source, (target / frame_name).open("wb") as dest:
                    shutil.copyfileobj(source, dest)
        zip_path.unlink()  # giữ dữ liệu đã giải nén, bỏ ZIP để tiết kiệm dung lượng
    csv_path = MAP_DIR / f"{video_id}.csv"
    if f"{video_id}.csv" in map_files and not csv_path.is_file():
        subprocess.run([rclone, "--config", str(rclone_config),
                        "--drive-root-folder-id", MAP_FOLDER_ID, "copyto",
                        f"iamtro:{video_id}.csv", str(csv_path), "-P"], check=True)
    print(video_id, len(list(target.glob("*.npy"))), "frames",
          "map" if csv_path.is_file() else "no map")
print("Embedding:", EMBED_DIR)
print("Map:", MAP_DIR)


In [ ]:
# 4. Build index bằng command hiện có trong repo
subprocess.run([sys.executable, "scripts/retrieval/run.py", "--task", "build-index",
                "--dataset", "v3c"], cwd=BACKEND_DIR, check=True)

import faiss
import pandas as pd
INDEX_DIR = REPO_DIR / "data/database/faiss_qwen3_vl_embedding_2b_vllm"
index = faiss.read_index(str(INDEX_DIR / "keyframes.faiss"))
metadata = pd.read_csv(INDEX_DIR / "metadata.csv", dtype={"video_id": str})
assert index.ntotal == len(metadata) > 0
assert set(metadata["video_id"]).issubset(set(ids)), "video_id trong metadata không đúng dạng 5 chữ số"
print(f"Đã index {index.ntotal:,} keyframes từ {metadata.video_id.nunique()} video")


In [ ]:
# 5. Cài và khởi động vLLM riêng; API dùng code của repo
import json
import time
import urllib.error

VLLM_VENV = WORK_DIR / "vllm-venv"
if not (VLLM_VENV / "bin/vllm").exists():
    subprocess.run([sys.executable, "-m", "uv", "venv", "--python", "3.12", str(VLLM_VENV)], check=True)
    subprocess.run([sys.executable, "-m", "uv", "pip", "install", "--python", str(VLLM_VENV / "bin/python"),
                    "--torch-backend=auto", "-r", str(BACKEND_DIR / "requirements-vllm.txt")], check=True)

def wait_http(url, seconds, ready):
    deadline = time.monotonic() + seconds
    while time.monotonic() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as response:
                payload = json.load(response)
            if ready(payload):
                return payload
        except (OSError, ValueError, urllib.error.URLError):
            pass
        time.sleep(3)
    raise TimeoutError(f"Chưa sẵn sàng: {url}")

vllm_log = WORK_DIR / "vllm.log"
vllm_handle = vllm_log.open("a", encoding="utf-8")
vllm_process = subprocess.Popen([str(VLLM_VENV / "bin/vllm"), "serve",
                                 "Qwen/Qwen3-VL-Embedding-2B", "--runner", "pooling",
                                 "--max-model-len", "8192", "--host", "127.0.0.1",
                                 "--port", "8001"], stdout=vllm_handle, stderr=subprocess.STDOUT)
try:
    wait_http("http://127.0.0.1:8001/v1/models", 600,
              lambda payload: any("Qwen3-VL-Embedding-2B" in item["id"] for item in payload.get("data", [])))
except Exception:
    print(vllm_log.read_text(encoding="utf-8")[-5000:])
    raise

api_log = WORK_DIR / "api.log"
api_handle = api_log.open("a", encoding="utf-8")
api_env = os.environ.copy()
api_env.update(QWEN_EMBEDDING_ENGINE="vllm", QWEN_VLLM_URL="http://127.0.0.1:8001/v1")
api_process = subprocess.Popen([sys.executable, "-m", "uvicorn", "src.api.main:app",
                                "--host", "127.0.0.1", "--port", "8000"],
                               cwd=BACKEND_DIR, env=api_env, stdout=api_handle,
                               stderr=subprocess.STDOUT)
try:
    health = wait_http("http://127.0.0.1:8000/", 180,
                       lambda payload: payload.get("retrieval_system_ready") is True)
except Exception:
    print(api_log.read_text(encoding="utf-8")[-5000:])
    raise
print("API health:", health)


In [ ]:
# 6. Mở tunnel ngrok cho frontend
from pyngrok import ngrok

ngrok_token = secret("NGROK_AUTHTOKEN")
if not ngrok_token:
    raise RuntimeError("Thiếu Kaggle Secret NGROK_AUTHTOKEN")
ngrok.set_auth_token(ngrok_token)
tunnel = ngrok.connect(8000, "http")
print("API URL:", tunnel.public_url)
print("Frontend VITE_API_BASE_URL=" + tunnel.public_url)
print("Kaggle cần giữ phiên notebook hoạt động để server tiếp tục phục vụ.")